# Stock versus PCP — notebook 127 setup, worker 0

Run workers 0 and 1 in separate L4 Colab runtimes. Each owns 200 disjoint historical episode identities, with one stock and one PCP rollout per identity. Total: 400 stock + 400 PCP episodes. No sweep and no other control arms.

Stock uses notebook 127's pure VLA sampler, historical batch reconstruction, render schedule, policy/VLM/tokenizer revisions, Torch/CUDA and stream-zero seeds. PCP uses frozen MC, normalized standardized RMS radius .02, three clean-action Q-gradient updates at sampler step 3 on every chunk, with P&P parent steps 1/2/3 K3/1/1. The first ten actions are corrected; flow then continues. There is no candidate reranking or separate re-noising cycle between those three gradient updates. This is the previously promising setting, so the comparison remains exploratory.

Both arms rerun under a fresh experiment `smolvla-pcp-stock-matched127-frozenmc-r002-k3-v1`. No requirement to match Jeff's 255 outcomes: notebook 127 is the accepted setup. Results, trajectories, batch identities, input/action hashes and correction telemetry are logged to Supabase. Final reporting includes rescues/spoils, conditional rates, sample counts, episode and task-group standard errors, and stock agreement with 127. Rerunning resumes saved batches.


In [ ]:
WORKER_INDEX = 0
BATCH_SIZE = 8
SIMULATOR_PROCESSES = 1  # parity runner uses Jeff's direct environment stepping
BUNDLE_PATH = ''
BUNDLE_BUCKET = 'artifacts'
BUNDLE_SHA256 = 'dd0f23bb04ef9302895e8fd654826b7015426e28e96022103310313a4da3945d'
BUNDLE_STORAGE_KEY = 'smolvla_pcp_sweep/bundles/dd0f23bb04ef9302895e8fd654826b7015426e28e96022103310313a4da3945d/trial_bundle.zip'
CHECKPOINT_SHA256 = {'frozen_mc': '9b8800bc9f27116afcb1812f93f8a9f6767ac012c7cdac790502032fbb7d0c8e'}


In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
os.environ.update(MUJOCO_GL='egl', NVIDIA_DRIVER_CAPABILITIES='compute,utility,graphics',
                  TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
from google.colab import userdata
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY','HF_TOKEN'):
    try:
        value=userdata.get(key)
        if value: os.environ[key]=value
    except Exception: pass
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY'):
    if not os.environ.get(key): raise RuntimeError(f'Add {key} in Colab Secrets and allow notebook access.')
if not BUNDLE_PATH:
    subprocess.run([sys.executable,'-m','pip','install','-q','supabase'],check=True)
    from supabase import create_client
    import hashlib
    client=create_client(os.environ['SUPABASE_URL'],os.environ['SUPABASE_SERVICE_KEY'])
    cached=Path('/content')/f'pcp_bundle_{BUNDLE_SHA256}.zip'
    if not cached.exists() or hashlib.sha256(cached.read_bytes()).hexdigest()!=BUNDLE_SHA256:
        data=client.storage.from_(BUNDLE_BUCKET).download(BUNDLE_STORAGE_KEY)
        if hashlib.sha256(data).hexdigest()!=BUNDLE_SHA256:
            raise RuntimeError('Downloaded bundle SHA256 mismatch')
        cached.write_bytes(data)
    BUNDLE_PATH=str(cached)
if __import__('hashlib').sha256(Path(BUNDLE_PATH).read_bytes()).hexdigest()!=BUNDLE_SHA256:
    raise RuntimeError('Bundle differs from the exact version pinned in this notebook')
PACKAGE=Path('/content/smolvla-pcp-trial')
PACKAGE.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise ValueError('Unsafe bundle path')
    archive.extractall(PACKAGE)
print({'python':sys.version,'executable':sys.executable},flush=True)
subprocess.run([sys.executable,'-m','pip','--version'],check=True)
install=subprocess.run([sys.executable,'-m','pip','install','-e',str(PACKAGE)+'[sim]',
                        'torch==2.11.0+cu128','torchvision==0.26.0+cu128',
                        '--extra-index-url','https://download.pytorch.org/whl/cu128'],
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
install_log=Path('/content/pcp_dependency_install.log')
install_log.write_text(install.stdout)
if install.returncode:
    print('\n'.join(install.stdout.splitlines()[-120:]),flush=True)
    raise RuntimeError(f'Dependency installation failed (exit {install.returncode}). '
                       f'Full log: {install_log}. Paste the ERROR lines above; no experiment has started.')
print('Simulation/model dependencies installed.',flush=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False,stdout=subprocess.DEVNULL)
if str(PACKAGE) not in sys.path: sys.path.insert(0,str(PACKAGE))
subprocess.run([sys.executable,'-m','pip','install',
                'torch==2.11.0+cu128','torchvision==0.26.0+cu128',
                '--index-url','https://download.pytorch.org/whl/cu128'],check=True)
if 'torch' in sys.modules and sys.modules['torch'].__version__ != '2.11.0+cu128':
    raise RuntimeError('Torch was already imported. Restart the runtime and rerun with the pinned version.')
from pnp import env_setup
if env_setup._find_nvidia_egl_library() is None:
    import re
    driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True).splitlines()[0]
    major=re.match(r'\d+',driver.strip()).group()
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',f'libnvidia-gl-{major}'],check=True)
env_setup._ensure_nvidia_egl_vendor()
if env_setup._find_nvidia_egl_library() is None: raise RuntimeError('NVIDIA EGL missing; hardware rendering is required.')
env_setup._ensure_libero_config()
env_setup._fix_torch_quant_compat()
env_setup._remove_broken_optional_torchaudio()
env_setup._ensure_libero_assets(token=os.getenv('HF_TOKEN'))
import torch, hashlib, json
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
metadata=json.loads((PACKAGE/'bundle_manifest.json').read_text())
if metadata['checkpoints']!=CHECKPOINT_SHA256: raise RuntimeError('Unexpected checkpoint manifest')
CHECKPOINT_PATHS={name:str(PACKAGE/'checkpoints'/f'{name}.pt') for name in metadata['checkpoints']}
for name,path in CHECKPOINT_PATHS.items():
    if hashlib.sha256(Path(path).read_bytes()).hexdigest()!=metadata['checkpoints'][name]:
        raise RuntimeError(f'Checkpoint mismatch: {name}')
print({'gpu':torch.cuda.get_device_name(0),'batch_size':BATCH_SIZE,'worker':WORKER_INDEX,'simulator_processes':SIMULATOR_PROCESSES})


In [ ]:
from pnp.smolvla_pcp_matched_trial import run_trial
report = run_trial(CHECKPOINT_PATHS['frozen_mc'], shard_index=0)
print(json.dumps(report,indent=2))


In [ ]:
from pnp.smolvla_pcp_matched_trial import comparison
print(json.dumps(comparison(),indent=2))
